# BB84 Post-processing Trace

This notebook starts after the quantum part has already produced sifted bits. The goal is to watch the classical part carefully enough that every number can be checked by hand.

We will follow one tiny 12-bit example through:

1. what sifting gives Alice and Bob,
2. QBER estimation,
3. removing public sample bits,
4. Cascade reconciliation,
5. hash-based verification,
6. privacy amplification.

The trace uses `step=...` numbers. These are teaching sequence numbers, not physical time. There is no fiber distance in this notebook because photons are no longer being simulated here.


In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
while not (repo_root / "examples").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from examples.postprocessing.bb84_event.cascade import CascadeController
from examples.postprocessing.bb84_event.helpers import parity, remove_positions, toeplitz_hash


class IdentityPermutationRNG:
    """Keep Cascade blocks in natural order for this hand-checkable tutorial."""

    def randint(self, start, stop):
        return stop


@dataclass(slots=True)
class TraceEvent:
    step_index: int
    actor: str
    step: str
    payload: dict
    receiver: str | None = None


@dataclass(slots=True)
class TeachingTrace:
    events: list[TraceEvent] = field(default_factory=list)
    step_index: int = 0

    def tick(self):
        self.step_index += 1

    def local(self, actor, step, payload):
        self.events.append(
            TraceEvent(
                step_index=self.step_index,
                actor=actor,
                receiver=None,
                step=step,
                payload=json.loads(json.dumps(payload)),
            )
        )

    def send(self, sender, receiver, message_type, payload):
        self.events.append(
            TraceEvent(
                step_index=self.step_index,
                actor=sender,
                receiver=receiver,
                step=message_type,
                payload=json.loads(json.dumps(payload)),
            )
        )

    def write_jsonl(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8") as stream:
            for event in self.events:
                stream.write(json.dumps({
                    "step_index": event.step_index,
                    "actor": event.actor,
                    "receiver": event.receiver,
                    "event": event.step,
                    "payload": event.payload,
                }, sort_keys=True, separators=(",", ":")))
                stream.write("\n")
        return path

    def pretty(self):
        lines = []
        for event in self.events:
            if event.receiver is None:
                header = f"step={event.step_index:02d} | {event.actor} local | {event.step}"
            else:
                header = f"step={event.step_index:02d} | {event.actor} -> {event.receiver} | {event.step}"
            lines.append(header)
            lines.append(f"       payload = {json.dumps(event.payload, sort_keys=True)}")
        return "\n".join(lines)


@dataclass(slots=True)
class TutorialResult:
    alice_initial_bits: list[int]
    bob_initial_bits: list[int]
    alice_reconciled_bits: list[int]
    bob_reconciled_bits: list[int]
    alice_final_key: list[int]
    bob_final_key: list[int]
    trace: TeachingTrace

    @property
    def final_keys_equal(self):
        return self.alice_final_key == self.bob_final_key


def run_tutorial_trace():
    trace = TeachingTrace()

    # Fixed tutorial input: these are already-sifted bits. Bob has one error at
    # index 6, so the later Cascade steps have something concrete to repair.
    alice_bits = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1]
    bob_bits = [1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1]
    teaching_error_indices = [
        index for index, (a, b) in enumerate(zip(alice_bits, bob_bits)) if a != b
    ]
    trace.local("setup", "sifted_inputs", {
        "alice_sifted_bits": alice_bits,
        "bob_sifted_bits": bob_bits,
        "known_teaching_error_indices": teaching_error_indices,
    })
    trace.tick()

    # Fixed tutorial sample: it avoids the known error, so QBER passes and
    # Cascade still has an error to find. In a full run this sample is random.
    sample_positions = [1, 5, 9]
    alice_sample_bits = [alice_bits[pos] for pos in sample_positions]
    trace.send("Alice", "Bob", "estimate.sample", {
        "sample_positions": sample_positions,
        "sample_bits": alice_sample_bits,
    })
    trace.tick()

    bob_sample_bits = [bob_bits[pos] for pos in sample_positions]
    sample_errors = sum(a != b for a, b in zip(alice_sample_bits, bob_sample_bits))
    qber = sample_errors / len(sample_positions)
    qber_abort_threshold = 0.11
    accept = qber <= qber_abort_threshold
    trace.local("Bob", "estimate.compute", {
        "bob_sample_bits_at_positions": bob_sample_bits,
        "alice_sample_bits_from_message": alice_sample_bits,
        "sample_errors": sample_errors,
        "qber": qber,
        "threshold": qber_abort_threshold,
        "accept": accept,
    })
    trace.send("Bob", "Alice", "estimate.result", {
        "qber": qber,
        "accept": accept,
        "sample_positions": sample_positions,
        "sample_size": len(sample_positions),
        "sample_errors": sample_errors,
    })
    trace.tick()

    alice_remaining = remove_positions(alice_bits, sample_positions)
    bob_remaining = remove_positions(bob_bits, sample_positions)
    remaining_positions = [
        index for index in range(len(alice_bits)) if index not in set(sample_positions)
    ]
    trace.local("Alice", "estimate.remove_public_sample", {
        "remaining_positions_from_sifted_key": remaining_positions,
        "remaining_bits": alice_remaining,
    })
    trace.local("Bob", "estimate.remove_public_sample", {
        "remaining_positions_from_sifted_key": remaining_positions,
        "remaining_bits": bob_remaining,
    })
    trace.tick()

    # Fixed tutorial Cascade settings: one pass and short blocks keep the parity
    # trace small enough to check by hand.
    cascade_passes = 1
    first_block_size = 5
    trace.send("Alice", "Bob", "cascade.start", {
        "qber": qber,
        "passes": cascade_passes,
        "first_block_size": first_block_size,
        "pass_indexing": "zero_based",
    })
    trace.tick()

    cascade = CascadeController(
        bits=bob_remaining,
        rng=IdentityPermutationRNG(),
        passes=cascade_passes,
        first_block_size=first_block_size,
    )

    while True:
        request = cascade.next_request()
        if request is None:
            if cascade.complete:
                break
            continue

        trace.send("Bob", "Alice", "cascade.parity_request", request.as_body())
        trace.tick()

        alice_parity = parity(alice_remaining, request.indices)
        trace.send("Alice", "Bob", "cascade.parity_response", {
            "request_id": request.request_id,
            "parity": alice_parity,
        })
        trace.tick()

        before_bits = bob_remaining.copy()
        bob_parity = parity(bob_remaining, request.indices)
        trace.local("Bob", "cascade.compare", {
            "request_id": request.request_id,
            "phase": request.phase,
            "query_indices": list(request.indices),
            "bob_local_parity": bob_parity,
            "alice_parity_from_message": alice_parity,
            "parity_mismatch": bob_parity != alice_parity,
        })
        cascade.apply_parity_response(
            request_id=request.request_id,
            alice_parity=alice_parity,
        )

        changed = [
            index for index, (old, new) in enumerate(zip(before_bits, bob_remaining))
            if old != new
        ]
        if changed:
            corrected = changed[0]
            trace.local("Bob", "cascade.correct_bit", {
                "remaining_key_index": corrected,
                "old_bit": before_bits[corrected],
                "new_bit": bob_remaining[corrected],
                "bob_bits_after_correction": bob_remaining,
            })
        trace.tick()

    trace.local("Bob", "cascade.complete", {
        "parity_requests": cascade.parity_requests,
        "corrections": cascade.corrections,
        "leaked_bits": cascade.leaked_bits,
        "alice_reconciled_bits": alice_remaining,
        "bob_reconciled_bits": bob_remaining,
        "reconciled_bits_equal": alice_remaining == bob_remaining,
    })
    trace.tick()

    # Fixed tutorial verification hash. The seed is public; the tag checks that
    # both sides now hold the same reconciled bits.
    verification_tag_len = 4
    verification_seed = [1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1]
    bob_tag = toeplitz_hash(bob_remaining, verification_seed, verification_tag_len)
    trace.send("Bob", "Alice", "verify.tag", {
        "tag_seed": verification_seed,
        "tag": bob_tag,
        "tag_len": verification_tag_len,
        "input_len": len(bob_remaining),
        "hash_family": "toeplitz",
    })
    trace.tick()

    alice_tag = toeplitz_hash(alice_remaining, verification_seed, verification_tag_len)
    verified = alice_tag == bob_tag
    trace.local("Alice", "verify.compute", {
        "alice_tag": alice_tag,
        "bob_tag_from_message": bob_tag,
        "verified": verified,
    })
    trace.send("Alice", "Bob", "verify.result", {"verified": verified})
    trace.tick()

    # Fixed tutorial privacy hash. The seed is public; the final key is printed
    # only because this notebook is a toy example.
    final_key_len = 4
    privacy_seed = [1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1]
    bob_final_key = toeplitz_hash(bob_remaining, privacy_seed, final_key_len)
    trace.local("Bob", "privacy.compute_local_key", {
        "toeplitz_seed": privacy_seed,
        "input_bits": bob_remaining,
        "final_key": bob_final_key,
    })
    trace.send("Bob", "Alice", "privacy.seed", {
        "toeplitz_seed": privacy_seed,
        "final_key_len": final_key_len,
        "input_len": len(bob_remaining),
        "hash_family": "toeplitz",
        "teaching_note": "seed is public; final key bits are shown only in this notebook",
    })
    trace.tick()

    alice_final_key = toeplitz_hash(alice_remaining, privacy_seed, final_key_len)
    trace.local("Alice", "privacy.compute_local_key", {
        "toeplitz_seed_from_message": privacy_seed,
        "input_bits": alice_remaining,
        "final_key": alice_final_key,
    })
    trace.send("Alice", "Bob", "finished", {"final_key_len": final_key_len})
    trace.tick()

    trace.local("summary", "done", {
        "alice_final_key": alice_final_key,
        "bob_final_key": bob_final_key,
        "final_keys_equal": alice_final_key == bob_final_key,
    })

    return TutorialResult(
        alice_initial_bits=alice_bits,
        bob_initial_bits=bob_bits,
        alice_reconciled_bits=alice_remaining,
        bob_reconciled_bits=bob_remaining,
        alice_final_key=alice_final_key,
        bob_final_key=bob_final_key,
        trace=trace,
    )


result = run_tutorial_trace()
events = result.trace.events


def find_event(name, *, actor=None, receiver=None):
    for event in events:
        if event.step != name:
            continue
        if actor is not None and event.actor != actor:
            continue
        if receiver is not None and event.receiver != receiver:
            continue
        return event
    raise ValueError(f"event not found: {name}")

print("Built the minimal BB84 post-processing trace inside this notebook.")
print("Number of trace records:", len(events))
print("Remember: step numbers show order, not physical time.")


## 1. What Sifting Gives Us

Sifting is the basis-matching step. Alice and Bob publicly compare bases, keep only matching-basis detections, and throw away the rest.

This notebook begins after that has happened. So the first lists below are already sifted bits. They are supposed to be almost equal, but not perfectly equal, because detector noise, channel noise, or other imperfections can flip some bits.


In [ ]:
alice_bits = result.alice_initial_bits
bob_bits = result.bob_initial_bits
mismatches = [i for i, (a, b) in enumerate(zip(alice_bits, bob_bits)) if a != b]

print("Alice sifted bits:", alice_bits)
print("Bob sifted bits:  ", bob_bits)
print("Mismatched sifted positions:", mismatches)
print("Number of sifted bits:", len(alice_bits))


## 2. QBER Estimation

QBER means quantum bit error rate. Alice reveals a few positions from the sifted key. Bob compares those public sample bits with his own bits at the same positions.

These sampled bits are no longer secret. After this step, both sides must remove them from the key material.


In [ ]:
estimate_sample = find_event("estimate.sample")
estimate_compute = find_event("estimate.compute", actor="Bob")

sample_positions = estimate_sample.payload["sample_positions"]
alice_sample_bits = estimate_sample.payload["sample_bits"]
bob_sample_bits = [bob_bits[pos] for pos in sample_positions]
sample_errors = sum(a != b for a, b in zip(alice_sample_bits, bob_sample_bits))
qber = sample_errors / len(sample_positions)

print("Alice -> Bob: estimate.sample")
print("Public sample positions:", sample_positions)
print("Alice sample bits:      ", alice_sample_bits)
print("Bob bits there:         ", bob_sample_bits)
print("Sample errors:", sample_errors)
print("QBER = errors / sample size =", sample_errors, "/", len(sample_positions), "=", qber)
print("Bob accepts this QBER:", estimate_compute.payload["accept"])


## 3. Remove The Public Sample

The sample bits were useful for estimating errors, but they were spoken in public. They cannot remain in the secret key.

Now Alice and Bob remove the sampled positions and continue with the remaining bits.


In [ ]:
alice_after_sample = find_event("estimate.remove_public_sample", actor="Alice")
bob_after_sample = find_event("estimate.remove_public_sample", actor="Bob")

remaining_positions = alice_after_sample.payload["remaining_positions_from_sifted_key"]
alice_remaining = alice_after_sample.payload["remaining_bits"]
bob_remaining_before_cascade = bob_after_sample.payload["remaining_bits"]
remaining_mismatches = [
    i for i, (a, b) in enumerate(zip(alice_remaining, bob_remaining_before_cascade)) if a != b
]

print("Remaining original sifted positions:", remaining_positions)
print("Alice remaining bits:", alice_remaining)
print("Bob remaining bits:  ", bob_remaining_before_cascade)
print("Mismatch positions in the remaining key:", remaining_mismatches)
print("Original sifted positions of those mismatches:", [remaining_positions[i] for i in remaining_mismatches])


## 4. Cascade Reconciliation

Cascade fixes Bob's remaining errors without Alice sending her whole key.

The basic question is simple: for a public set of positions, Alice sends one parity bit. Bob computes his parity for the same positions. If the parities differ, that block contains an odd number of errors. Then Bob asks smaller parity questions until the wrong bit is isolated and flipped.


In [ ]:
print("Cascade messages and local comparisons")
print("--------------------------------------")
for event in events:
    if not event.step.startswith("cascade"):
        continue
    if event.receiver is None:
        direction = f"{event.actor} local"
    else:
        direction = f"{event.actor} -> {event.receiver}"
    print(f"step={event.step_index:02d} | {direction} | {event.step}")
    print("  ", event.payload)


The first Cascade request is enough to see the idea. Alice and Bob compute parity on the same indices. The result is different, so the block must contain an error.


In [ ]:
first_request = find_event("cascade.parity_request")
first_response = find_event("cascade.parity_response")
indices = first_request.payload["indices"]

alice_p = parity(alice_remaining, indices)
bob_p = parity(bob_remaining_before_cascade, indices)

print("First Cascade query indices:", indices)
print("Alice bits at those indices:", [alice_remaining[i] for i in indices])
print("Bob bits at those indices:  ", [bob_remaining_before_cascade[i] for i in indices])
print("Alice parity:", alice_p)
print("Bob parity:  ", bob_p)
print("Parity mismatch:", alice_p != bob_p)
print("Alice's response message carried parity:", first_response.payload["parity"])


In [ ]:
correction = find_event("cascade.correct_bit")
cascade_done = find_event("cascade.complete")
corrected_remaining_index = correction.payload["remaining_key_index"]
corrected_original_index = remaining_positions[corrected_remaining_index]

print("Cascade corrected remaining-key index:", corrected_remaining_index)
print("That maps back to original sifted index:", corrected_original_index)
print("Old Bob bit:", correction.payload["old_bit"])
print("New Bob bit:", correction.payload["new_bit"])
print("Alice reconciled bits:", cascade_done.payload["alice_reconciled_bits"])
print("Bob reconciled bits:  ", cascade_done.payload["bob_reconciled_bits"])
print("Reconciled bits equal:", cascade_done.payload["reconciled_bits_equal"])
print("Parity bits leaked during Cascade:", cascade_done.payload["leaked_bits"])


## 5. Verification Hash

After Cascade, Bob thinks the keys match. Verification checks that without revealing the key.

Bob sends a public hash seed and a short tag of his reconciled bits. Alice computes the same hash on her reconciled bits. If the tags match, they accept the reconciled key.


In [ ]:
verify_tag = find_event("verify.tag")
verify_compute = find_event("verify.compute")

seed = verify_tag.payload["tag_seed"]
tag_len = verify_tag.payload["tag_len"]
bob_tag = verify_tag.payload["tag"]

alice_recomputed_tag = toeplitz_hash(result.alice_reconciled_bits, seed, tag_len)
bob_recomputed_tag = toeplitz_hash(result.bob_reconciled_bits, seed, tag_len)

print("Bob -> Alice: verify.tag")
print("Public Toeplitz seed:", seed)
print("Bob tag in message:", bob_tag)
print("Alice recomputed tag:", alice_recomputed_tag)
print("Bob recomputed tag:  ", bob_recomputed_tag)
print("Verification accepted:", verify_compute.payload["verified"])


## 6. Privacy Amplification Hash

Cascade and verification leaked some public information. Privacy amplification compresses the reconciled bits into a shorter key.

The hash seed is public. The secrecy comes from hashing private reconciled bits down to a shorter final key.


In [ ]:
privacy_seed_message = find_event("privacy.seed")
alice_privacy = find_event("privacy.compute_local_key", actor="Alice")
bob_privacy = find_event("privacy.compute_local_key", actor="Bob")
summary = find_event("done", actor="summary")

privacy_seed = privacy_seed_message.payload["toeplitz_seed"]
final_key_len = privacy_seed_message.payload["final_key_len"]

alice_final = toeplitz_hash(result.alice_reconciled_bits, privacy_seed, final_key_len)
bob_final = toeplitz_hash(result.bob_reconciled_bits, privacy_seed, final_key_len)

print("Bob -> Alice: privacy.seed")
print("Public Toeplitz seed:", privacy_seed)
print("Final key length:", final_key_len)
print("Alice final key:", alice_final)
print("Bob final key:  ", bob_final)
print("Final keys equal:", summary.payload["final_keys_equal"])
print("This notebook shows the key only because it is a tiny teaching example.")


## 7. One Useful Failure Check

A good way to understand the verification hash is to break one bit and watch the tag change. This is not a protocol step; it is a quick sanity check.


In [ ]:
broken_bob_bits = result.bob_reconciled_bits.copy()
broken_bob_bits[0] ^= 1
broken_tag = toeplitz_hash(broken_bob_bits, seed, tag_len)

print("Correct Bob tag:", bob_tag)
print("Tag after flipping Bob reconciled bit 0:", broken_tag)
print("Would verification pass after this flip:", broken_tag == alice_recomputed_tag)


## 8. Full Trace

Here is the complete compact trace. It is the same run we inspected above, but shown as a message log.


In [ ]:
trace_path = repo_root / "tutorials" / "bb84_postprocessing" / "minimal_postprocessing_trace.jsonl"
result.trace.write_jsonl(trace_path)

print(result.trace.pretty())
print()
print("Saved JSONL trace to:", trace_path)
